# 02 — LSTM + FastText Indonesia (PyTorch)

Model revisi dari `LSTM_GloVe_Final__with_GloVe_.ipynb`. Perubahan utama dari versi asli:

| Aspek | Versi asli (GloVe) | Versi revisi (notebook ini) |
|---|---|---|
| Framework | TensorFlow/Keras | **PyTorch** |
| Word embedding | GloVe 100d **Bahasa Inggris** (`trainable=False`) | **FastText Indonesia resmi** (`cc.id.300.bin`, 300d, subword-aware, `trainable=True`) |
| Urutan split vs balancing | SMOTE **sebelum** split (leakage) | **Split dulu**, baru balancing hanya di training set |
| Penanganan imbalance | SMOTE pada token ID | **Class-weighted loss** |
| Panjang sekuens | `max_len` = panjang maksimum di seluruh dataset | `max_len` = persentil ke-95 training set + truncation |

Lihat `01_data_audit.ipynb` untuk detail & alasan setiap perubahan.

## 1. Instalasi & Setup

Notebook ini didesain untuk Google Colab (GPU runtime disarankan: Runtime -> Change runtime type -> GPU) atau dijalankan pada lokal terintgrasi GPU. Cell di bawah menginstall semua package yang dibutuhkan, termasuk Pytorch dan library resmi fastText dari Facebook Research.

> ⚠️ **Kalau menjalankan di lokal (bukan Colab), baca dulu 2 catatan penting di cell instalasi
> di bawah** — ada 2 titik yang sering gagal di environment lokal (terutama Windows) tapi
> biasanya otomatis beres di Colab: versi PyTorch yang cocok dengan CUDA driver kamu, dan
> build package `fasttext` di Windows.

In [ ]:
# ============================================================================
# CATATAN PENTING kalau menjalankan di LOKAL (bukan Colab) -- baca dulu:
#
# 1) PyTorch + CUDA
#    Baris `pip install torch torchvision torchaudio` polos di bawah ini SERING
#    menginstall build CPU-only kalau resolver pip tidak otomatis mendeteksi CUDA
#    kamu dengan benar -- ini jebakan paling umum di lokal, tidak muncul di Colab
#    (Colab sudah punya PyTorch+CUDA pre-installed & konsisten).
#
#    Cara paling aman untuk GPU seperti RTX 3060 Ti:
#      a. Cek versi CUDA yang didukung driver kamu:  nvidia-smi   (lihat baris "CUDA Version")
#      b. Buka https://pytorch.org/get-started/locally/ , pilih OS=Windows, Package=Pip,
#         Compute Platform = versi CUDA yang muncul di nvidia-smi (atau yang terdekat di
#         bawahnya) -- situs itu akan generate command index-url yang tepat, misalnya:
#         pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
#      c. Jalankan command hasil dari situs itu SEBAGAI GANTI baris `!pip install torch...`
#         generik di bawah (baris itu sengaja saya biarkan sebagai fallback untuk Colab).
#
# 2) Package `fasttext` sering GAGAL build dari source di Windows (perlu Microsoft C++
#    Build Tools terpasang). Kalau `pip install fasttext` error "Failed building wheel",
#    pakai fork dengan wheel Windows prebuilt sebagai gantinya (API-nya identik,
#    tetap `import fasttext`):
#      pip install fasttext-wheel
#    (bukan `pip install fasttext`)
#
# 3) Disarankan pakai `%pip install` (magic command), bukan `!pip install` (shell command),
#    supaya package pasti terinstall ke kernel Jupyter yang sedang aktif -- di lokal, sering
#    ada beberapa instalasi Python/pip di PATH sekaligus yang bisa bikin `!pip` menginstall
#    ke Python yang salah. Cell di bawah sudah pakai `%pip` untuk alasan ini.
#
# 4) Sangat disarankan jalankan di virtual environment terpisah (venv/conda), bukan
#    Python sistem -- supaya tidak bentrok dengan environment project lain.
# ============================================================================

# PyTorch -- baris fallback generik ini OK untuk Colab. Untuk lokal, ganti/tambahkan
# flag --index-url sesuai Catatan #1 di atas SEBELUM menjalankan cell ini.
%pip install -q torch torchvision torchaudio

# FastText -- kalau di Windows lokal dan baris ini error, ganti jadi:
#   %pip install -q fasttext-wheel
%pip install -q fasttext

# Utilitas lain
%pip install -q scikit-learn pandas numpy matplotlib seaborn tqdm


In [2]:
# Verifikasi PyTorch benar-benar mendeteksi GPU sebelum lanjut -- kalau hasilnya
# 'CUDA available: False' padahal kamu punya RTX 3060 Ti, itu tandanya PyTorch
# yang ter-install adalah build CPU-only (lihat Catatan #1 di cell instalasi di atas).
import torch as _torch_check
print("PyTorch version :", _torch_check.__version__)
print("CUDA available  :", _torch_check.cuda.is_available())
if _torch_check.cuda.is_available():
    print("GPU device      :", _torch_check.cuda.get_device_name(0))
    print("VRAM (GB)       :", round(_torch_check.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("\n⚠️  GPU TIDAK terdeteksi -- training akan sangat lambat di CPU.")
    print("    Kemungkinan besar PyTorch ter-install versi CPU-only. Lihat Catatan #1")
    print("    di cell instalasi sebelumnya untuk cara install ulang versi CUDA yang benar.")


PyTorch version : 2.12.1+cu126
CUDA available  : True
GPU device      : NVIDIA GeForce RTX 3060 Ti
VRAM (GB)       : 8.6


In [4]:
import os
import re
import json
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


## 2. Download FastText Indonesia resmi

Kita pakai model **`.bin`** (bukan sekadar `.vec`), karena `.bin` menyimpan informasi
**subword (character n-gram)** — ini keunggulan utama FastText dibanding GloVe/Word2Vec:
kata yang TIDAK ada persis di kosakata pretrained (mis. bentuk hasil stemming seperti `"libat"`
dari `"melibatkan"`) tetap bisa mendapat vektor yang masuk akal, dikomposisikan dari potongan
n-gram karakternya — bukan otomatis jadi vektor nol seperti kasus GloVe sebelumnya.

> ⚠️ File model resmi `cc.id.300.bin` cukup besar (beberapa GB terkompresi). Proses download +
> ekstrak bisa memakan waktu. Pastikan runtime Colab kamu punya cukup disk space (biasanya
> aman, disk Colab ~100GB).

In [9]:
import fasttext
import fasttext.util

# Bagi yang belum download library fastText, Download resmi dari Facebook Research (otomatis skip kalau sudah pernah didownload)
# fasttext.util.download_model('id', if_exists='ignore')  # 'id' = kode bahasa Indonesia

# Bagi yang sudah download library fastText, load model bahasa Indonesia (cc.id.300.bin) dari file lokal
FASTTEXT_MODEL_PATH = r'../model/cc.id.300.bin'
ft_model = fasttext.load_model(FASTTEXT_MODEL_PATH)

print("FastText model loaded.")
print("Embedding dimension:", ft_model.get_dimension())
print("Contoh vocabulary size (kata eksplisit, di luar subword):", len(ft_model.get_words()))


FastText model loaded.
Embedding dimension: 300
Contoh vocabulary size (kata eksplisit, di luar subword): 2000000


In [10]:
# Sanity check cepat: FastText harus bisa memberi vektor untuk kata stemmed Indonesia
# yang mungkin tidak persis ada di kosakata (dibuktikan lewat subword composition)
sample_words = ['prabowo', 'jokowi', 'libat', 'koalisi', 'dukung', 'kpk', 'ganjar']
for w in sample_words:
    vec = ft_model.get_word_vector(w)
    in_vocab = w in ft_model.get_words()
    print(f"{w:12s} | ada di vocab eksplisit: {str(in_vocab):5s} | norma vektor: {np.linalg.norm(vec):.3f}")


prabowo      | ada di vocab eksplisit: True  | norma vektor: 0.791
jokowi       | ada di vocab eksplisit: True  | norma vektor: 0.966
libat        | ada di vocab eksplisit: True  | norma vektor: 0.675
koalisi      | ada di vocab eksplisit: True  | norma vektor: 0.760
dukung       | ada di vocab eksplisit: True  | norma vektor: 0.807
kpk          | ada di vocab eksplisit: True  | norma vektor: 1.271
ganjar       | ada di vocab eksplisit: True  | norma vektor: 0.643


**Catatan penting:** semua kata di atas akan tetap mendapat vektor dengan norma > 0
(bukan nol), **bahkan untuk kata yang tidak ada di daftar vocabulary eksplisit** — ini
konfirmasi langsung bahwa FastText benar-benar mengatasi masalah *frozen zero-vector* yang
kita temukan di notebook GloVe lama.

## 3. Load & siapkan data

Ingat dari `01_data_audit.ipynb`: teks di kolom `desc` **sudah bersih & sudah di-stem** —
kita tidak mengulang cleaning, hanya melakukan tokenization + numericalization untuk PyTorch.

In [11]:
df = pd.read_csv('../data/dataset_merge_v2.csv')
df = df.drop_duplicates(subset=['desc']).reset_index(drop=True)
print("Shape setelah drop duplikat:", df.shape)
print(df['label'].value_counts())

Shape setelah drop duplikat: (9532, 5)
label
VALID    8233
HOAX     1299
Name: count, dtype: int64


In [12]:
# Tokenisasi sederhana: split whitespace (sudah cukup karena data sudah bersih dari tanda baca)
def simple_tokenize(text):
    return str(text).split()

df['tokens'] = df['desc'].apply(simple_tokenize)
df['length'] = df['tokens'].apply(len)

print(df['length'].describe())

count    9532.000000
mean      207.995069
std       148.424341
min        31.000000
25%       140.750000
50%       179.000000
75%       231.000000
max      3530.000000
Name: length, dtype: float64


## 4. Split train/val/test — SEBELUM penanganan imbalance

Perbaikan kunci dari versi lama: split dilakukan di sini, di awal, sebelum keputusan
penanganan-imbalance apa pun dibuat. Val & test set murni representasi distribusi data asli
(termasuk ketidakseimbangan kelasnya) — tidak boleh ada informasi dari val/test yang bocor ke
proses balancing atau ke pembangunan vocabulary.

In [15]:
X = df['tokens'].tolist()
y = df['label'].map({'VALID': 0, 'HOAX': 1}).values

# 70% train, 20% validation, 10% test -- sama seperti proporsi di paper asli
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=1/3, random_state=SEED, stratify=y_temp
)

print(f"Train : {len(X_train)} ({np.bincount(y_train)})")
print(f"Val   : {len(X_val)} ({np.bincount(y_val)})")
print(f"Test  : {len(X_test)} ({np.bincount(y_test)})")


Train : 6672 ([5763  909])
Val   : 1906 ([1646  260])
Test  : 954 ([824 130])


## 5. Tentukan `max_len` dari training set (bukan seluruh dataset)

Sesuai temuan Bagian 4 di notebook audit: pakai persentil ke-95 panjang teks **training set
saja**, bukan panjang maksimum absolut. ini mengurangi disparitas jumlah padding antar kelas
yang berpotensi jadi shortcut trivial.

In [16]:
train_lengths = [len(t) for t in X_train]
max_len = int(np.percentile(train_lengths, 95)) # ambil 95th percentile sebagai max_len
print(f"Percentile ke-95 panjang teks (training set): {max_len} token")
print(f"(Sebagai perbandingan, panjang maksimum absolut di training set: {max(train_lengths)} token)")

Percentile ke-95 panjang teks (training set): 379 token
(Sebagai perbandingan, panjang maksimum absolut di training set: 2764 token)


## 6. Bangun vocabulary (dari training set saja)

In [17]:
from collections import Counter

MIN_FREQ = 2
PAD_TOKEN, UNK_TOKEN = '<PAD>', '<UNK>'

counter = Counter()
for tokens in X_train:
    counter.update(tokens)

vocab_words = [w for w, c in counter.items() if c >= MIN_FREQ]
vocab_words = sorted(vocab_words, key=lambda w: -counter[w])

word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}

for w in vocab_words:
    word2idx[w] =  len(word2idx)
    
vocab_size = len(word2idx)
print(f"Ukuran vocabulary (min freq={MIN_FREQ}, dari training set): {vocab_size}")

Ukuran vocabulary (min freq=2, dari training set): 28707


## 7. Bangun embedding matrix dari FastText

Berbeda dari versi GloVe lama (kata tak-ditemukan → vektor nol & frozen selamanya), di sini
**setiap kata di vocabulary — termasuk kata hasil stemming yang tidak baku — tetap mendapat
vektor dari FastText** lewat komposisi subword. Layer embedding juga di-set trainable
(`requires_grad=True`, default di PyTorch), supaya model masih bisa menyesuaikan representasi
selama training.

In [ ]:
EMBED_DIM = ft_model.get_dimension() # 300

embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype=np.float32)
oov_via_subword = 0

for word, idx in word2idx.items():
    if word in (PAD_TOKEN,):
        continue # tetap vektor nol khusus padding
    if word == UNK_TOKEN:
        embedding_matrix[idx] = np.random.normal(scale=0.1, size=EMBED_DIM)
        continue
    vec = ft_model.get_word_vector(word) # selalu mengembalikan vektor, walau OOV (via subword)
    embedding_matrix[idx] = vec
    if word not in ft_model.get_words():
        oov_via_subword += 1

print(f"Total kata di vocabulary: {vocab_size}")
print(f"Kata yang didapat lewa komposisi subword (tidak ada persis di vocab FastText): {oov_via_subword} ({oov_via_subword/vocab_size*100:.1f}%)")
print(f"Shape embedding matrix: {embedding_matrix.shape}")